In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
df_check = pd.read_csv("../disdata/sinca_data/sinca_RVIII874_wd_020701_230630.csv", sep=";", encoding="latin1")
print(df_check.columns.tolist())
df_check.head()

FileNotFoundError: [Errno 2] No such file or directory: '../disdata/sinca_data/sinca_RVIII874_wd_020701_230630.csv'

In [ ]:
from pyproj import Transformer

zone_number = 18
hemisphere = "south"

transformer = Transformer.from_crs(
    f"EPSG:{32700 + zone_number if hemisphere == 'south' else 32600 + zone_number}",
    "EPSG:4326",
    always_xy=True
)

easting = 674983
northing =  5912937

lon, lat = transformer.transform(easting, northing)
print(f"Longitude: {lon}, Latitude: {lat}")

In [ ]:
MASTER_COLS = ["date", "site", "longitude", "latitude",
               "PM10", "PM25", "NOX", "NO2", "NO", "O3", "SO2",
               "temp", "ws", "wd"]

FILENAME_TAG_MAP = {
    "pm10": "PM10", "pm25": "PM25",
    "nox": "NOX", "no2": "NO2", "no": "NO",
    "o3": "O3", "so2": "SO2",
    "temp": "temp", "ws": "ws", "wd": "wd",
}

def extract_pollutant_from_filename(filepath):
    fname = os.path.basename(filepath).lower()
    parts = fname.replace(".csv", "").split("_")
    for part in parts:
        if part in FILENAME_TAG_MAP:
            return FILENAME_TAG_MAP[part]
    return None

def load_raw_file(filepath):
    df = pd.read_csv(filepath, sep=";", encoding="latin1", low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    date_col = df.columns[0]
    hour_col = df.columns[1]
    value_cols = list(df.columns[2:])

    df["datetime"] = pd.to_datetime(
        df[date_col].astype(str).str.zfill(6) + df[hour_col].astype(str).str.zfill(4),
        format="%y%m%d%H%M",
        errors="coerce"
    )

    value = pd.Series(np.nan, index=df.index, dtype="float64")
    for col in value_cols:
        numeric = pd.to_numeric(df[col], errors="coerce")
        value = value.fillna(numeric)

    pollutant = extract_pollutant_from_filename(filepath)
    return pd.DataFrame({"datetime": df["datetime"], pollutant: value}), pollutant

def combine_station(folder, station_code_prefix, site_code, longitude, latitude,
                     expected_start, expected_end):
    files = glob.glob(f"{folder}/sinca_{station_code_prefix}_*.csv")
    print(f"Found {len(files)} files for {site_code}: {files}")

    merged = None
    for f in files:
        df, pollutant = load_raw_file(f)
        if pollutant is None:
            print(f"  WARNING: could not identify pollutant from filename: {f}")
            continue
        merged = df if merged is None else pd.merge(merged, df, on="datetime", how="outer")

    for col in ["PM10", "PM25", "NOX", "NO2", "NO", "O3", "SO2", "temp", "ws", "wd"]:
        if col not in merged.columns:
            merged[col] = np.nan

    merged["site"] = site_code
    merged["longitude"] = longitude
    merged["latitude"] = latitude
    merged = merged.sort_values("datetime")

    start_dt = pd.to_datetime(expected_start, format="%d/%m/%Y")
    end_dt = pd.to_datetime(expected_end, format="%d/%m/%Y") + pd.Timedelta(hours=23)
    before_trim = len(merged)
    merged = merged[(merged["datetime"] >= start_dt) & (merged["datetime"] <= end_dt)]
    print(f"{site_code}: trimmed from {before_trim} to {len(merged)} rows within {expected_start} to {expected_end}")

    merged["date"] = merged["datetime"].dt.strftime("%d/%m/%Y %H:%M")
    return merged[MASTER_COLS]

# ---- Run for one station ----
rviii833 = combine_station(
    folder="../disdata/sinca_data/sinca7stations_raw",
    station_code_prefix="RVIII833",     # replace with the actual filename prefix for this station
    site_code="RVIII/833",              # Cerro Merquín's code
    longitude=-73.03570483450228,                     # fill in real coordinates
    latitude=-36.91335897483023,
    expected_start="01/07/2002",
    expected_end="30/06/2023"
)
rviii833.to_csv("../disdata/sinca_data/sinca_RVIII833_combined.csv", index=False)

In [ ]:
# load the existing combined file for this station
filepath = "../disdata/sinca_data/sinca_RVIII833_combined.csv"  # fill in the actual filename
df = pd.read_csv(filepath)

df["datetime"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M")

# build the full expected hourly grid
start_dt = pd.to_datetime("01/07/2002", format="%d/%m/%Y") + pd.Timedelta(hours=1)
end_dt = pd.to_datetime("30/06/2023", format="%d/%m/%Y") + pd.Timedelta(hours=23)
full_range = pd.date_range(start_dt, end_dt, freq="h")

# preserve site/longitude/latitude before reindexing (they'd otherwise become NaN in the new rows)
site_val = df["site"].iloc[0]
lon_val = df["longitude"].iloc[0]
lat_val = df["latitude"].iloc[0]

df = df.set_index("datetime").reindex(full_range)
df.index.name = "datetime"
df = df.reset_index()

# refill the metadata columns across the newly added rows
df["site"] = site_val
df["longitude"] = lon_val
df["latitude"] = lat_val

df["date"] = df["datetime"].dt.strftime("%d/%m/%Y %H:%M")

print(f"Reindexed to {len(df)} rows, spanning {df['date'].iloc[0]} to {df['date'].iloc[-1]}")

MASTER_COLS = ["date", "site", "longitude", "latitude",
               "PM10", "PM25", "NOX", "NO2", "NO", "O3", "SO2",
               "temp", "ws", "wd"]
df = df[MASTER_COLS]

df.to_csv(filepath, index=False)

In [ ]:
import pandas as pd
import glob
import os

def sense_check_station(folder, station_code_prefix, combined_filepath):
    print(f"=== Checking {station_code_prefix} ===")
    
    # 1. Load the combined file
    combined = pd.read_csv(combined_filepath)
    print(f"Combined file: {len(combined)} rows, columns: {combined.columns.tolist()}")
    
    # 2. Confirm date range matches expectation
    combined_dates = pd.to_datetime(combined["date"], format="%d/%m/%Y %H:%M")
    print(f"Combined date range: {combined_dates.min()} to {combined_dates.max()}")
    
    # 3. Check row count matches an even hourly grid across the trimmed period
    expected_hours = len(pd.date_range(combined_dates.min(), combined_dates.max(), freq="h"))
    print(f"Expected hourly rows for this range: {expected_hours}, actual: {len(combined)}")
    
    # 4. Compare each pollutant column's non-null count against its raw source file
    raw_files = glob.glob(f"{folder}/sinca_{station_code_prefix}_*.csv")
    for f in raw_files:
        fname = os.path.basename(f).lower()
        for tag in ["pm10", "pm25", "nox", "no2", "no", "o3", "so2", "temp", "ws", "wd"]:
            if f"_{tag}_" in fname:
                pollutant = tag.upper() if tag in ["pm10", "pm25", "nox", "no2", "so2"] else tag
                pollutant = "PM25" if tag == "pm25" else pollutant
                
                raw_df = pd.read_csv(f, sep=";", encoding="latin1", low_memory=False)
                raw_df.columns = [c.strip() for c in raw_df.columns]
                value_cols = raw_df.columns[2:]
                raw_nonnull = raw_df[value_cols].notna().any(axis=1).sum()
                
                combined_col = [c for c in combined.columns if c.lower() == pollutant.lower()]
                if combined_col:
                    combined_nonnull = combined[combined_col[0]].notna().sum()
                    match = "OK" if abs(raw_nonnull - combined_nonnull) <= 1 else "MISMATCH"
                    print(f"  {pollutant}: raw non-null={raw_nonnull}, combined non-null={combined_nonnull} [{match}]")
    
    # 5. Spot-check a handful of actual values against the raw file directly
    print()
    return combined

# ---- Run for each of your 6 completed stations ----
stations = [
    ("RVIII831", "../disdata/sinca_data/sinca_RVIII831_combined.csv"),
    ("RVIII874", "../disdata/sinca_data/sinca_RVIII874_combined.csv"),
    ("RVIII810", "../disdata/sinca_data/sinca_RVIII810_combined.csv"),
    ("RVIII833", "../disdata/sinca_data/sinca_RVIII833_combined.csv"),
    ("RVIII850", "../disdata/sinca_data/sinca_RVIII850_combined.csv"),
    ("RVIII852", "../disdata/sinca_data/sinca_RVIII852_combined.csv"),
]

for prefix, combined_path in stations:
    sense_check_station("../disdata/sinca_data", prefix, combined_path)